# Data Generator - Synthetic Accident Data
This notebook generates realistic accident data with intentional noise injection.

In [ ]:
# Cell 1: Imports
import pandas as pd
import random
import sys
import os

# Add 'src' to path so we can import our scripts
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from generator_utils import get_time_of_day, get_traffic_density, get_weather, determine_actual_severity, generate_description

In [ ]:
# Cell 2: Configuration
NUM_RECORDS = 10000
road_types = ['Highway', 'Urban', 'Rural']
vehicles = ['Sedan', 'Truck', 'SUV', 'Motorcycle']
data = []

print(f"Generating {NUM_RECORDS} records...")

In [ ]:
# Cell 3: Generation Loop
for i in range(NUM_RECORDS):
    # 1. Generate Context
    hour = get_time_of_day()
    road = random.choice(road_types)
    weather = get_weather()
    speed_limit = 100 if road == 'Highway' else (60 if road == 'Rural' else 40)
    actual_speed = int(random.normalvariate(speed_limit, 10)) 
    traffic = get_traffic_density(hour, road)
    vehicle = random.choice(vehicles)
    
    # 2. Determine "Ground Truth" (Physics)
    actual_severity = determine_actual_severity(hour, weather, actual_speed, road)
    
    # 3. Generate Description
    desc = generate_description(actual_severity, weather, vehicle)
    
    # 4. INJECT NOISE (The "Lie")
    # Rule: 15% of MAJOR accidents are misreported as MINOR by humans
    reported_severity = actual_severity
    if actual_severity == 1:
        if random.random() < 0.15:  # 15% chance to lie
            reported_severity = 0 
    
    # 5. Save Row
    data.append({
        'Record_ID': i,
        'Hour_of_Day': hour,
        'Road_Type': road,
        'Weather': weather,
        'Traffic_Density': traffic,
        'Speed_Limit': speed_limit,
        'Incident_Description': desc,
        'Reported_Severity': reported_severity, 
        'Actual_Severity': actual_severity      
    })

print(f"Generation complete!")

In [ ]:
# Cell 4: Save Data
df = pd.DataFrame(data)
output_path = '../data/raw/synthetic_accident_data_v1.csv'
df.to_csv(output_path, index=False)

print(f"Data Generated! Saved to {output_path}")
print(df.head())